In [16]:
import random
import time
from IPython.display import clear_output
from collections import deque

In [22]:
class Cell:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.walls = [True, True, True, True]  # top, right, bottom, left
        self.visited = False
        self.is_start = False
        self.is_target = False

class Maze:
    def __init__(self, width, height, num_targets=1, is_image=False):
        if num_targets < 1:
            raise ValueError("num_targets must be at least 1")

        self.width = width
        self.height = height
        self.num_targets = num_targets
        self.start = None
        self.targets = []
        self.grid = [[Cell(x, y) for y in range(height)] for x in range(width)]
        if not is_image:
            self.generate()

    def generate(self):
        stack = []
        current_cell = random.choice(random.choice(self.grid))
        current_cell.visited = True

        while True:
            neighbors = self.get_unvisited_neighbors(current_cell)
            if neighbors:
                next_cell = random.choice(neighbors)
                self.remove_wall(current_cell, next_cell)
                stack.append(current_cell)
                current_cell = next_cell
                current_cell.visited = True
            elif stack:
                current_cell = stack.pop()
            else:
                break

        for x in range(self.width):
            for y in range(self.height):
                if random.random() < 0.1:
                    cell = self.grid[x][y]
                    directions = [(0, -1), (1, 0), (0, 1), (-1, 0)]
                    valid = [(dx, dy) for dx, dy in directions
                             if 0 <= x + dx < self.width and 0 <= y + dy < self.height]
                    if valid:
                        dx, dy = random.choice(valid)
                        neighbor = self.grid[x + dx][y + dy]
                        self.remove_wall(cell, neighbor)

        self.assign_start_and_targets()

    def assign_start_and_targets(self):
        min_dist = (self.width + self.height) / 4
        all_cells = [self.grid[x][y] for x in range(self.width) for y in range(self.height)]

        for cell in all_cells:
            cell.is_start = False
            cell.is_target = False

        for start in random.sample(all_cells, len(all_cells)):
            valid_targets = [
                cell for cell in all_cells
                if cell is not start and self.manhattan_distance(start, cell) >= min_dist
            ]
            if len(valid_targets) >= self.num_targets:
                self.start = start
                self.targets = random.sample(valid_targets, self.num_targets)
                self.start.is_start = True
                for target in self.targets:
                    target.is_target = True
                return

        raise ValueError("Could not place the requested number of targets far enough from the start.")

    def manhattan_distance(self, first, second):
        return abs(first.x - second.x) + abs(first.y - second.y)

    def get_unvisited_neighbors(self, cell):
        neighbors = []
        directions = [(0, -1), (1, 0), (0, 1), (-1, 0)]  # top, right, bottom, left
        for dx, dy in directions:
            nx, ny = cell.x + dx, cell.y + dy
            if 0 <= nx < self.width and 0 <= ny < self.height:
                neighbor = self.grid[nx][ny]
                if not neighbor.visited:
                    neighbors.append(neighbor)
        return neighbors

    def remove_wall(self, current, next):
        dx = next.x - current.x
        dy = next.y - current.y
        if dx == 1:  # next is to the right
            current.walls[1] = False
            next.walls[3] = False
        elif dx == -1:  # next is to the left
            current.walls[3] = False
            next.walls[1] = False
        elif dy == 1:  # next is below
            current.walls[2] = False
            next.walls[0] = False
        elif dy == -1:  # next is above
            current.walls[0] = False
            next.walls[2] = False

    def display(self):
        for y in range(self.height):
            # Print the top walls
            for x in range(self.width):
                if self.grid[x][y].walls[0]:
                    print("+---", end="")
                else:
                    print("+   ", end="")
            print("+")
            # Print the left walls and cell contents
            for x in range(self.width):
                wall = "|" if self.grid[x][y].walls[3] else " "
                cell = self.grid[x][y]
                if cell.is_start:
                    print(f"{wall} O ", end="")
                elif cell.is_target:
                    print(f"{wall} X ", end="")
                else:
                    print(f"{wall}   ", end="")
            print("|")
        # Print the bottom walls of the last row
        for x in range(self.width):
            if self.grid[x][self.height - 1].walls[2]:
                print("+---", end="")
            else:
                print("+   ", end="")
        print("+")

In [24]:
class Agent:
    def __init__(self, maze):
        self.reset(maze)

    def reset(self, maze):
        self.maze = maze
        self.position = maze.start
        self.maze_solution = None
        self.img_x = 0
        self.img_y = 0
        self.maze_img = Maze(1, 1, is_image=True)
        self.maze_img.grid[0][0].walls = list(self.maze.start.walls)
        self.maze_img.grid[0][0].is_start = True
        self.maze_img.grid[0][0].visited = True

    def DFS(self, sleep_time=0.15):
        self.reset(self.maze)
        stack = []
        # path stores image cells so coordinates stay in the same system
        path = [self.maze_img.grid[self.img_x][self.img_y]]

        # Seed the stack with the start cell's unvisited neighbors
        valid_directions = [i for i, w in enumerate(self.position.walls) if not w]
        for direction in valid_directions:
            dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][direction]
            self.add_row_or_col_to_img(direction)
            neighbor = self.maze_img.grid[self.img_x + dx][self.img_y + dy]
            if not neighbor.visited:
                stack.append(neighbor)
                neighbor.visited = True

        while stack:
            next_cell = stack.pop()

            if not (self.img_x == next_cell.x and self.img_y == next_cell.y):
                # Backtrack along path until we can reach next_cell in one step
                while not self.can_reach_cell(next_cell):
                    path.pop()  # remove current from path
                    prev_cell = path[-1]  # peek at the previous cell
                    self.move_in_img(prev_cell.x - self.img_x, prev_cell.y - self.img_y, sleep_time=sleep_time)
                # Now adjacent — move to next_cell
                self.move_in_img(next_cell.x - self.img_x, next_cell.y - self.img_y, sleep_time=sleep_time)

            path.append(self.maze_img.grid[self.img_x][self.img_y])

            if self.position.is_target:
                self.maze_solution = path
                clear_output(wait=False)
                self.display_maze_img(show_solution=True)
                print("Reached a target!")
                return

            valid_directions = [i for i, w in enumerate(self.position.walls) if not w]
            for direction in valid_directions:
                dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][direction]
                self.add_row_or_col_to_img(direction)
                neighbor = self.maze_img.grid[self.img_x + dx][self.img_y + dy]
                if not neighbor.visited:
                    stack.append(neighbor)
                    neighbor.visited = True

    def BFS(self, sleep_time=0.15):
        self.reset(self.maze)
        queue = deque([self.position])      # real-maze cells discovered (in BFS tree)
        discovered = {self.position}        # cells added to the queue
        explored = {self.position}          # cells the robot has physically sensed
        parent = {self.position: None}      # BFS tree for path reconstruction

        while queue:
            current_cell = queue.popleft()

            # Robot physically walks from its current position to `current_cell`,
            # only travelling through cells it has already explored.
            # `current_cell` itself may be unexplored — we step onto it from its
            # explored neighbour (its BFS parent).
            if current_cell is not self.position:
                nav_path = self._bfs_navigate(self.position, current_cell, explored)
                for step in nav_path[1:]:
                    dx = step.x - self.position.x
                    dy = step.y - self.position.y
                    direction = {(0, -1): 0, (1, 0): 1, (0, 1): 2, (-1, 0): 3}[(dx, dy)]
                    self.add_row_or_col_to_img(direction)
                    self.move_in_img(dx, dy, sleep_time=sleep_time)
                explored.add(current_cell)

            if current_cell.is_target:
                # Reconstruct the shortest path through the BFS tree
                real_path = []
                node = current_cell
                while node is not None:
                    real_path.append(node)
                    node = parent[node]
                real_path.reverse()

                self.maze_solution = self._real_to_img_path(real_path)

                clear_output(wait=False)
                self.display_maze_img(show_solution=True)
                print("Reached a target!")
                return

            # Sense walls at the current cell; queue any undiscovered neighbours.
            for i, w in enumerate(current_cell.walls):
                if w:
                    continue
                dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][i]
                neighbor = self.maze.grid[current_cell.x + dx][current_cell.y + dy]
                if neighbor not in discovered:
                    discovered.add(neighbor)
                    parent[neighbor] = current_cell
                    queue.append(neighbor)

    def find_all_targets(self, sleep_time=0.15):
        self.reset(self.maze)
        stack = []
        path = [self.maze_img.grid[self.img_x][self.img_y]]
        found_targets = []
        found_target_set = set()
        discovery_log = []

        if self.position.is_target:
            found_targets.append(self.position)
            found_target_set.add(self.position)
            discovery_log.append(f"Found target 1 at ({self.position.x}, {self.position.y})")

        valid_directions = [i for i, w in enumerate(self.position.walls) if not w]
        for direction in valid_directions:
            dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][direction]
            self.add_row_or_col_to_img(direction)
            neighbor = self.maze_img.grid[self.img_x + dx][self.img_y + dy]
            if not neighbor.visited:
                stack.append(neighbor)
                neighbor.visited = True

        while stack:
            next_cell = stack.pop()

            if not (self.img_x == next_cell.x and self.img_y == next_cell.y):
                while not self.can_reach_cell(next_cell):
                    path.pop()
                    prev_cell = path[-1]
                    self.move_in_img(prev_cell.x - self.img_x, prev_cell.y - self.img_y, sleep_time=sleep_time)
                self.move_in_img(next_cell.x - self.img_x, next_cell.y - self.img_y, sleep_time=sleep_time)

            path.append(self.maze_img.grid[self.img_x][self.img_y])

            if self.position.is_target and self.position not in found_target_set:
                found_targets.append(self.position)
                found_target_set.add(self.position)
                discovery_log.append(
                    f"Found target {len(found_targets)} at ({self.position.x}, {self.position.y})"
                )

            valid_directions = [i for i, w in enumerate(self.position.walls) if not w]
            for direction in valid_directions:
                dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][direction]
                self.add_row_or_col_to_img(direction)
                neighbor = self.maze_img.grid[self.img_x + dx][self.img_y + dy]
                if not neighbor.visited:
                    stack.append(neighbor)
                    neighbor.visited = True

        optimal_real_path = self._link_targets_optimally(found_targets)
        self.maze_solution = self._real_to_img_path(optimal_real_path)

        clear_output(wait=False)
        self.display_maze_img(show_solution=True)
        for log_entry in discovery_log:
            print(log_entry)
        print(f"Linked {len(found_targets)} targets in {len(optimal_real_path) - 1} steps.")
        return found_targets, optimal_real_path

    def _bfs_navigate(self, start, goal, explored):
        """Shortest path from start to goal that travels only through `explored`
        cells. The goal itself may be unexplored — it's reached by stepping out
        from an explored neighbour."""
        q = deque([start])
        came_from = {start: None}
        while q:
            cell = q.popleft()
            if cell is goal:
                path = [cell]
                while came_from[cell] is not None:
                    cell = came_from[cell]
                    path.append(cell)
                path.reverse()
                return path
            for i, w in enumerate(cell.walls):
                if w:
                    continue
                dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][i]
                neighbor = self.maze.grid[cell.x + dx][cell.y + dy]
                if neighbor in came_from:
                    continue
                if neighbor is goal or neighbor in explored:
                    came_from[neighbor] = cell
                    q.append(neighbor)
        return None

    def _get_open_neighbors(self, cell):
        neighbors = []
        for i, wall in enumerate(cell.walls):
            if wall:
                continue
            dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][i]
            neighbors.append(self.maze.grid[cell.x + dx][cell.y + dy])
        return neighbors

    def _shortest_path_between(self, start, goal):
        queue = deque([start])
        came_from = {start: None}

        while queue:
            current = queue.popleft()
            if current is goal:
                path = [current]
                while came_from[current] is not None:
                    current = came_from[current]
                    path.append(current)
                path.reverse()
                return path

            for neighbor in self._get_open_neighbors(current):
                if neighbor in came_from:
                    continue
                came_from[neighbor] = current
                queue.append(neighbor)

        return None

    def _link_targets_optimally(self, targets):
        if not targets:
            return [self.maze.start]

        points = [self.maze.start] + targets
        pair_paths = {}
        pair_distances = {}

        for i in range(len(points)):
            for j in range(i + 1, len(points)):
                path = self._shortest_path_between(points[i], points[j])
                if path is None:
                    raise ValueError("Maze graph is disconnected; could not link all targets.")
                pair_paths[(i, j)] = path
                pair_paths[(j, i)] = list(reversed(path))
                pair_distances[(i, j)] = len(path) - 1
                pair_distances[(j, i)] = len(path) - 1

        target_count = len(targets)
        dp = {}

        for target_index in range(target_count):
            mask = 1 << target_index
            dp[(mask, target_index)] = (pair_distances[(0, target_index + 1)], None)

        full_mask = (1 << target_count) - 1
        for mask in range(1, full_mask + 1):
            for last in range(target_count):
                if (mask, last) not in dp:
                    continue
                current_distance, _ = dp[(mask, last)]
                for next_target in range(target_count):
                    if mask & (1 << next_target):
                        continue
                    next_mask = mask | (1 << next_target)
                    candidate_distance = current_distance + pair_distances[(last + 1, next_target + 1)]
                    state = (next_mask, next_target)
                    if state not in dp or candidate_distance < dp[state][0]:
                        dp[state] = (candidate_distance, last)

        best_last = min(
            range(target_count),
            key=lambda last: dp[(full_mask, last)][0]
        )

        ordered_target_indices = []
        mask = full_mask
        last = best_last
        while last is not None:
            ordered_target_indices.append(last)
            _, previous = dp[(mask, last)]
            mask ^= 1 << last
            last = previous
        ordered_target_indices.reverse()

        route = [self.maze.start]
        previous_point_index = 0
        for target_index in ordered_target_indices:
            current_point_index = target_index + 1
            route.extend(pair_paths[(previous_point_index, current_point_index)][1:])
            previous_point_index = current_point_index
        return route

    def _real_to_img_cell(self, cell):
        offset_x = self.position.x - self.img_x
        offset_y = self.position.y - self.img_y
        return self.maze_img.grid[cell.x - offset_x][cell.y - offset_y]

    def _real_to_img_path(self, real_path):
        return [self._real_to_img_cell(cell) for cell in real_path]

    def can_reach_cell(self, target):
        dx, dy = target.x - self.img_x, target.y - self.img_y
        if abs(dx) + abs(dy) == 1:
            if dx == 1 and not self.position.walls[1]:  # right
                return True
            elif dx == -1 and not self.position.walls[3]:  # left
                return True
            elif dy == 1 and not self.position.walls[2]:  # down
                return True
            elif dy == -1 and not self.position.walls[0]:  # up
                return True
        return False

    def random_explore(self, steps=25, sleep_time=0.15):
        self.reset(self.maze)
        for step in range(steps):
            valid_directions = [i for i, w in enumerate(self.position.walls) if not w]
            next_direction = random.choice(valid_directions)
            dx, dy = [(0, -1), (1, 0), (0, 1), (-1, 0)][next_direction]

            # Expand image if agent is at the edge in that direction
            self.add_row_or_col_to_img(next_direction)

            self.move_in_img(dx, dy, sleep_time=sleep_time)
            print(f"Step {step + 1}/{steps}")

    def move_in_img(self, dx, dy, sleep_time=0.15):
        # Move in actual maze and image
        self.position = self.maze.grid[self.position.x + dx][self.position.y + dy]
        self.img_x += dx
        self.img_y += dy

        # Reveal the new cell in the image
        img_cell = self.maze_img.grid[self.img_x][self.img_y]
        img_cell.walls = list(self.position.walls)
        img_cell.visited = True
        img_cell.is_start = self.position.is_start
        img_cell.is_target = self.position.is_target
        # Propagate open passages to already-existing neighbors in the image
        for di, (ndx, ndy) in enumerate([(0, -1), (1, 0), (0, 1), (-1, 0)]):
            nx, ny = self.img_x + ndx, self.img_y + ndy
            if 0 <= nx < self.maze_img.width and 0 <= ny < self.maze_img.height:
                if not img_cell.walls[di]:
                    opposite = [2, 3, 0, 1][di]
                    self.maze_img.grid[nx][ny].walls[opposite] = False

        clear_output(wait=True)
        self.display_maze_img()
        time.sleep(sleep_time)

    def reveal_shared_walls(self, direction):
        if direction == 0:  # New row at top (y=0); existing cells now at y=1+
            for x in range(self.maze_img.width):
                neighbor = self.maze_img.grid[x][1]
                if neighbor.visited and not neighbor.walls[0]:
                    self.maze_img.grid[x][0].walls[2] = False
        elif direction == 1:  # New column at right (x=width-1)
            for y in range(self.maze_img.height):
                neighbor = self.maze_img.grid[self.maze_img.width - 2][y]
                if neighbor.visited and not neighbor.walls[1]:
                    self.maze_img.grid[self.maze_img.width - 1][y].walls[3] = False
        elif direction == 2:  # New row at bottom (y=height-1)
            for x in range(self.maze_img.width):
                neighbor = self.maze_img.grid[x][self.maze_img.height - 2]
                if neighbor.visited and not neighbor.walls[2]:
                    self.maze_img.grid[x][self.maze_img.height - 1].walls[0] = False
        elif direction == 3:  # New column at left (x=0); existing cells now at x=1+
            for y in range(self.maze_img.height):
                neighbor = self.maze_img.grid[1][y]
                if neighbor.visited and not neighbor.walls[3]:
                    self.maze_img.grid[0][y].walls[1] = False

    def add_row_or_col_to_img(self, direction):
        if direction == 0:  # Add row at top
            if self.img_y != 0:
                return
            for col in self.maze_img.grid:
                col.insert(0, Cell(0, 0))
            self.maze_img.height += 1
            self.reveal_shared_walls(0)
            self.img_y += 1
        elif direction == 1:  # Add column at right
            if self.img_x != self.maze_img.width - 1:
                return
            self.maze_img.grid.append([Cell(0, 0) for _ in range(self.maze_img.height)])
            self.maze_img.width += 1
            self.reveal_shared_walls(1)
        elif direction == 2:  # Add row at bottom
            if self.img_y != self.maze_img.height - 1:
                return
            for col in self.maze_img.grid:
                col.append(Cell(0, 0))
            self.maze_img.height += 1
            self.reveal_shared_walls(2)
        elif direction == 3:  # Add column at left
            if self.img_x != 0:
                return
            self.maze_img.grid.insert(0, [Cell(0, 0) for _ in range(self.maze_img.height)])
            self.maze_img.width += 1
            self.reveal_shared_walls(3)
            self.img_x += 1

        for x in range(self.maze_img.width):
            for y in range(self.maze_img.height):
                self.maze_img.grid[x][y].x = x
                self.maze_img.grid[x][y].y = y

    def display_maze_img(self, show_solution=False):
        solution_set = {(c.x, c.y) for c in self.maze_solution} if (show_solution and self.maze_solution) else set()
        for y in range(self.maze_img.height):
            for x in range(self.maze_img.width):
                if self.maze_img.grid[x][y].walls[0]:
                    print("+---", end="")
                else:
                    print("+   ", end="")
            print("+")
            for x in range(self.maze_img.width):
                wall = "|" if self.maze_img.grid[x][y].walls[3] else " "
                cell = self.maze_img.grid[x][y]
                if not show_solution and x == self.img_x and y == self.img_y:
                    print(f"{wall} A ", end="")
                elif cell.is_start:
                    print(f"{wall} O ", end="")
                elif cell.is_target:
                    if not cell.visited:
                        print(f"{wall} ? ", end="")
                    else:
                        print(f"{wall} X ", end="")
                elif (x, y) in solution_set:
                    print(f"{wall} ■ ", end="")
                elif not cell.visited:
                    print(f"{wall} ? ", end="")
                else:
                    print(f"{wall}   ", end="")
            right_wall = "|" if self.maze_img.grid[self.maze_img.width - 1][y].walls[1] else " "
            print(right_wall)
        for x in range(self.maze_img.width):
            if self.maze_img.grid[x][self.maze_img.height - 1].walls[2]:
                print("+---", end="")
            else:
                print("+   ", end="")
        print("+")

In [27]:
maze = Maze(5, 5, 2)
maze.display()
agent = Agent(maze)

+---+---+---+---+---+
|       |     X     |
+   +   +   +---+   +
|   |   |   |       |
+   +---+   +   +   +
|         O |   |   |
+   +---+---+   +   +
|           |       |
+   +   +---+---+   +
|   |             X |
+---+---+---+---+---+


In [28]:
# Agent randomly explores the maze for n steps
# sleep_time is the time the code pauses between each step (speed of the visualization)
# agent.random_explore(steps=100, sleep_time=0.05)

In [29]:
# Agent performs a Depth First Search to find a target
# sleep_time is the time the code pauses between each step (speed of the visualization)
agent.DFS(sleep_time=0.1)

+---+---+---+---+---+
|   | ? |   | ? | ? |
+   +---+   +---+---+
| ■   ■   O | ? | ? |
+   +---+---+---+---+
| ■   ■     | ? | ? |
+   +   +---+---+   +
|   | ■   ■   ■   X |
+---+---+---+---+---+
Reached a target!


In [30]:
# Agent performs a Breadth First Search to find a target
# sleep_time is the time the code pauses between each step (speed of the visualization)
agent.BFS(sleep_time=0.05)

+---+---+---+---+
| ? | ? | ■   X  
+---+---+   +---+
| ? | ? | ■ | ? |
+   +---+   +---+
|         O | ? |
+   +---+---+---+
Reached a target!


In [31]:
# Find and connect all targets with a globally shortest route from the start
found_targets, optimal_path = agent.find_all_targets(sleep_time=0.05)

+---+---+---+---+---+
|       | ■   X   ■ |
+   +   +   +---+   +
|   |   | ■ |     ■ |
+   +---+   +   +   +
|         O |   | ■ |
+   +---+---+   +   +
|           |     ■ |
+   +   +---+---+   +
|   |             X |
+---+---+---+---+---+
Found target 1 at (4, 4)
Found target 2 at (3, 0)
Linked 2 targets in 8 steps.
